Step 1. 데이터 준비 및 정제
•	라벨링 최적화: 7,500장의 정면 사진을 5개 감정(Neutral, Happiness, Anger, Sadness, Panic)으로 재분류.
•	보류 데이터 처리: 모호한 표정들을 별도로 모아 다수결로 재정의하여 모델의 판단 경계 보강.
-> 라벨링 교체 완료
-> BOX이상 데이터 제거 작업 필요함

In [18]:
ADRESS = '/workspace/user5/workspace/user5/img'

img_adress = f'{ADRESS}/img(classify)'
label_adress = f'{ADRESS}/relabeled' # Corrected 'iabel' to 'label'
# label_adress = '/workspace/user1/data_classification/modified'
segmentation_adress = f'{ADRESS}/segmentation'

In [19]:
img_adress

'/workspace/user5/workspace/user5/img/img(classify)'

In [20]:
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageOps # Import ImageOps for exif_transpose
import matplotlib.pyplot as plt
import random
import json # Ensure json is imported for loading .json files
from collections import Counter # For mode calculation
from scipy.stats import mode # For mode calculation
import matplotlib.patches as patches # For drawing bounding boxes

# Assuming ADRESS, img_adress, label_adress, segmentation_adress are already defined.

# 이 부분을 건드려서 해당 파트의 파일만 확인 가능
###########################################################################################
# EMOTIONS = ['anger', 'happy', 'panic', 'sadness', 'neutral']
EMOTIONS = ['anger', 'happy', 'panic', 'sadness', 'blank']
DATA_TYPES_TO_PROCESS = ['train', 'val'] # As per the request, focus on 'train' and 'val' for now
###########################################################################################

def load_emotion_data_files(emotion, data_type):
    """
    Loads JSON label data and NPZ segmentation data for a given emotion and data type.
    Returns (json_data_list, npz_data_dict) or (None, None) if files not found.
    """
    json_file_path = os.path.join(label_adress, data_type, f'{data_type}_{emotion}_modified.json') # Corrected 'iabel_adress' to 'label_adress'
    npz_file_path = os.path.join(segmentation_adress, data_type, f'{data_type}_{emotion}.npz')

    current_json_data = None
    current_npz_data = None

    # Load JSON data (using cp949 encoding as identified in previous steps)
    if os.path.exists(json_file_path):
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                current_json_data = json.load(f)
            print(f"Loaded JSON for {emotion} ({data_type}) from {json_file_path}")
        except Exception as e:
            print(f"Error loading JSON for {emotion} ({data_type}) from {json_file_path}: {e}")
    else:
        print(f"JSON file not found: {json_file_path}")

    # Load NPZ data
    if os.path.exists(npz_file_path):
        try:
            current_npz_data = np.load(npz_file_path)
            print(f"Loaded NPZ for {emotion} ({data_type}) from {npz_file_path}")
        except Exception as e:
            print(f"Error loading NPZ for {emotion} ({data_type}) from {npz_file_path}: {e}")
    else:
        print(f"NPZ file not found: {npz_file_path}")

    return current_json_data, current_npz_data

In [ ]:
# Test Data전용으로 작성한 코드. uploader만 5가지 분류로 재 배치한 것.

'''
def combined_data_iterator():
    """
    지정한 data_type과 emotion에 대해
    이미지, 라벨, segmentation mask가 모두 매칭되는 항목만 하나씩 yield하는 generator.
    """
    for data_type in DATA_TYPES_TO_PROCESS:
        for emotion in EMOTIONS:
            # 해당 감정/데이터타입의 JSON과 NPZ를 불러옴
            json_labels, npz_segments = load_emotion_data_files(emotion, data_type)

            # 둘 중 하나라도 없으면 이 조합은 건너뜀
            if json_labels is None or npz_segments is None:
                print(f"Skipping {emotion} {data_type} due to missing JSON or NPZ data.")
                if npz_segments is not None:
                    npz_segments.close()  # 열려 있으면 닫기
                continue

            # 이미지 폴더 경로 구성
            # 가정: img_adress/data_type/emotion/filename.jpg
            img_emotion_dir = os.path.join(img_adress, data_type, emotion)

            # 이미지 폴더가 없으면 건너뜀
            if not os.path.exists(img_emotion_dir):
                print(f"Image directory not found: {img_emotion_dir}. Skipping this emotion for {data_type}.")
                if npz_segments is not None:
                    npz_segments.close()
                continue

            # JSON에 들어있는 각 샘플을 하나씩 순회
            for json_entry in json_labels:
                # 현재 샘플의 파일명 가져오기
                filename = json_entry.get('filename')
                if not filename:
                    continue  # filename이 없으면 건너뜀

                # 실제 이미지 파일 경로
                img_full_path = os.path.join(img_emotion_dir, filename)

                # NPZ에서 같은 이름의 segmentation mask 가져오기
                segmentation_mask = npz_segments.get(filename)

                # 이미지 파일이 실제로 존재하고, mask도 있으면 매칭 성공
                if os.path.exists(img_full_path) and segmentation_mask is not None:
                    # 기본 메타데이터를 entry_data에 저장
                    entry_data = {
                        'filename': filename,
                        'emotion': emotion,  # 어떤 감정 폴더에서 왔는지 저장
                        'data_type': data_type,  # train/val/test 같은 구분 저장
                        'gender': json_entry.get('gender'),
                        'age': json_entry.get('age'),
                        'isProf': json_entry.get('isProf'),
                        'faceExp_uploader': json_entry.get('faceExp_uploader'),
                        'bg_uploader': json_entry.get('bg_uploader'),
                    }

                    # -------------------------------------------------
                    # annot_A / annot_B / annot_C 라벨을 모두 수집
                    # 각 annotator가 가진 faceExp, bg, boxes를 저장
                    # -------------------------------------------------
                    for annotator in ['A', 'B', 'C']:
                        annot_key = f'annot_{annotator}'

                        # annot_A 같은 키가 있고, 값이 dict면 읽기
                        if annot_key in json_entry and isinstance(json_entry[annot_key], dict):
                            entry_data[f'faceExp_{annotator}'] = json_entry[annot_key].get('faceExp')
                            entry_data[f'bg_{annotator}'] = json_entry[annot_key].get('bg')
                            entry_data[f'boxes_{annotator}'] = json_entry[annot_key].get('boxes')
                        else:
                            # 없으면 None으로 채움
                            entry_data[f'faceExp_{annotator}'] = None
                            entry_data[f'bg_{annotator}'] = None
                            entry_data[f'boxes_{annotator}'] = None

                    # segmentation mask와 이미지 경로도 저장
                    entry_data['segmentation_mask'] = segmentation_mask
                    entry_data['img_path'] = img_full_path

                    # 이 항목을 하나씩 밖으로 내보냄
                    yield entry_data

            # 해당 emotion 처리가 끝나면 npz 파일 닫기
            if npz_segments is not None:
                npz_segments.close()


# 제너레이터 생성
all_matched_data_gen = combined_data_iterator()

# ==========================================
# (이전 코드 생략: combined_data_iterator 함수 정의까지 동일)
# ==========================================

# 제너레이터 생성
all_matched_data_gen = combined_data_iterator()

# 메모리 초과 방지를 위해 무거운 배열(segmentation_mask)을 제외하고 메타데이터만 추출
metadata_list = []
for entry in all_matched_data_gen:
    # 딕셔너리 컴프리헨션을 사용하여 segmentation_mask 키만 제외
    meta = {k: v for k, v in entry.items() if k != 'segmentation_mask'}
    metadata_list.append(meta)

# 추출된 메타데이터를 Pandas DataFrame으로 변환
df = pd.DataFrame(metadata_list)

# DataFrame을 CSV 파일로 로컬에 중간 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
csv_save_path = '/workspace/user1/model/step1_metadata.csv' 
df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')

# 결과 확인
print(f"메모리 최적화 완료: 총 {len(df)}건의 메타데이터가 '{csv_save_path}'에 저장되었습니다.")
'''

JSON file not found: /workspace/user5/workspace/user5/img/relabeled/train/train_anger_modified.json
Loaded NPZ for anger (train) from /workspace/user5/workspace/user5/img/segmentation/train/train_anger.npz
Skipping anger train due to missing JSON or NPZ data.
JSON file not found: /workspace/user5/workspace/user5/img/relabeled/train/train_happy_modified.json
Loaded NPZ for happy (train) from /workspace/user5/workspace/user5/img/segmentation/train/train_happy.npz
Skipping happy train due to missing JSON or NPZ data.
JSON file not found: /workspace/user5/workspace/user5/img/relabeled/train/train_panic_modified.json
Loaded NPZ for panic (train) from /workspace/user5/workspace/user5/img/segmentation/train/train_panic.npz
Skipping panic train due to missing JSON or NPZ data.
JSON file not found: /workspace/user5/workspace/user5/img/relabeled/train/train_sadness_modified.json
Loaded NPZ for sadness (train) from /workspace/user5/workspace/user5/img/segmentation/train/train_sadness.npz
Skipping

Step 2. 극장 환경 시뮬레이션 (Augmentation)
•	저조도/저화질 구현: 학습 시 실시간으로 감마 보정(어둡게), 가우시안 노이즈, 블러(뭉개짐)를 적용.
•	강건성 확보: 모델이 깨끗한 사진뿐만 아니라 악조건의 사진에서도 동일한 특징을 추출하도록 유도.


In [9]:
import cv2
import torch
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

# 1. 극장 환경 시뮬레이션을 위한 Augmentation 파이프라인 정의
theater_transform = A.Compose([
    # 저조도 (어둡게): gamma_limit이 낮을수록 어두워짐
    A.RandomGamma(gamma_limit=(50, 90), p=0.8), 
    
    # 고감도 노이즈: 어두운 곳에서 촬영 시 발생하는 센서 노이즈
    A.GaussNoise(var_limit=(20.0, 60.0), p=0.5), 
    
    # 초점 흐림: 동적 움직임이나 초점 나감 현상 (뭉개짐)
    A.Blur(blur_limit=5, p=0.3), 
    
    # 모델 입력 사이즈 맞춤 (CNN/ViT 규격에 따라 변경 가능)
    A.Resize(224, 224), 
    
    # 정규화 및 PyTorch 텐서 변환
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 2. CSV 기반의 커스텀 Dataset 클래스
class TheaterEmotionDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        # Step 1에서 만든 메타데이터 CSV 로드 (메모리 절약)
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        
        # 감정 문자열을 정수형 라벨로 변환하기 위한 매핑
        self.label_map = {
            'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3, 'neutral': 4
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = row['img_path']
        label_str = row['faceExp_uploader']
        
        # 라벨 매핑 (CSV에 저장된 감정 값 기준)
        label = self.label_map.get(label_str, -1) 

        # OpenCV로 이미지 로드
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"이미지를 찾을 수 없습니다: {img_path}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Augmentation 적용
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

# 3. DataLoader 인스턴스화 및 실행 준비
csv_file_path = '/workspace/user1/model/step1_metadata.csv'

# 데이터셋 객체 생성
train_dataset = TheaterEmotionDataset(csv_path=csv_file_path, transform=theater_transform)

# DataLoader 설정
train_loader = DataLoader(
    train_dataset, 
    batch_size=32,       # GPU 메모리에 맞게 32~64 권장
    shuffle=True, 
    num_workers=4,       # CPU 멀티프로세싱으로 이미지 로드 속도 향상
    pin_memory=True      # GPU로의 데이터 전송 속도 향상
)

# 정상 로드 확인용 코드
if __name__ == "__main__":
    print(f"전체 데이터 개수: {len(train_dataset)}")
    images, labels = next(iter(train_loader))
    print(f"Batch Image Tensor Shape: {images.shape}")
    print(f"Batch Labels Shape: {labels.shape}")

/tmp/ipykernel_166985/4007964245.py:14: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(20.0, 60.0), p=0.5),


EmptyDataError: No columns to parse from file

Step 3. 이원화 모델 학습 (A40 서버 활용)
•	Model A (CNN 기반): ResNet50 + DAN. 이목구비의 미세한 변화(Attention)를 포착하여 감정 분류.
•	Model B (Transformer 기반): ViT-Base. 이미지 전체의 맥락과 분위기를 파악하여 감정 분류.
•	최적화: 대용량 VRAM(48GB)을 활용해 높은 배치 사이즈와 정규화 전략 적용.
•   Fine tunning작업:Augmentation 된 추가 데이터로 Fine tunning작업. mAP지표 기반 Pretrained model의 성능을 재분배한 라벨링 한 이미지의 방향으로 가중치 변형

Step 4. 감정 데이터 추출 및 비교
•	Top 3 추출: 단순 1등 감정이 아닌, 상위 3개 감정의 이름과 확률값(Confidence)을 리포트.
•	모델 밸런스 평가: 저화질 환경에서 어느 모델이 더 안정적으로 Top 3 분포를 내뱉는지 비교 분석.


Step 5. 감정 합성 (Synthesis) 준비
•	벡터 저장: 각 모델이 뽑아낸 특징 벡터(Feature Vector)와 Top 3 확률을 매칭하여 저장.
•	합성 로직: 추출된 확률 비중을 가중치로 삼아, 생성 모델(GAN/Diffusion)을 통해 새로운 '복합 감정 표정' 생성.
